# Spark configuration

Spark configuration —  This block creates a Spark session configured with Iceberg and MinIO.
It enables the lakehouse catalog so that all tables can be read and written through Iceberg.

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:2025-12-31 ships Spark 4.1.0 — print spark.version to confirm.

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")

    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")

Spark 4.1.0   catalog: lakehouse


DataFrame[]

# Load zone lookup table

Load zone lookup table  — This loads the static NYC Taxi Zone lookup table, which will later be used to enrich trip records with pickup and dropoff zone names.

In [2]:
zones = spark.read.parquet("data/taxi_zone_lookup.parquet")
zones.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


# Reading from Kafka

Reading from Kafka — Here we define a streaming source that reads raw messages from the Kafka topic taxi-trips.
The stream includes Kafka metadata (key, value, partition, offset, timestamp) and will be used as the input for the Bronze layer.

In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC     = "taxi-trips"

raw_stream = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", BOOTSTRAP)
        .option("subscribePattern", "taxi-trips-.*")
        .option("startingOffsets", "earliest")
        .load()
)


# Bronze Layer — Raw Kafka Ingestion (Custom Scenario)

- Read from the Kafka topic using Structured Streaming.
- Write raw events as-is to a bronze Iceberg table.
- Configure checkpointing so that restarting the job does not produce duplicates.

Bronze layer write  — This section writes the raw Kafka messages into an Iceberg Bronze table.
The data is stored exactly as received, and checkpointing ensures the stream can restart without duplicating records.

In [4]:
print("isStreaming:", raw_stream.isStreaming)   # True — not a batch DataFrame
print()
raw_stream.printSchema()

isStreaming: True

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [5]:
# Parse the value column from binary → JSON → individual fields.
# The schema must be declared explicitly; inference is not available for streams.

event_schema = "VendorID INTEGER, tpep_pickup_datetime STRING, tpep_dropoff_datetime STRING, passenger_count STRING, trip_distance STRING, RatecodeID STRING, store_and_fwd_flag STRING, PULocationID STRING, DOLocationID STRING, payment_type INTEGER, fare_amount INTEGER, extra STRING, mta_tax INTEGER, tip_amount INTEGER, tolls_amount INTEGER, improvement_surcharge STRING, total_amount INTEGER, congestion_surcharge INTEGER, airport_fee INTEGER, cbd_congestion_fee INTEGER"

parsed_stream = (
    raw_stream
    .select(
        F.col("key").cast("string").alias("key"),
        F.from_json(F.col("value").cast("string"), event_schema).alias("d"),
        "partition",
        "offset",
        F.col("timestamp").alias("kafka_time"),
    )
    .select("key", "d.*", "partition", "offset", "kafka_time")
)

print("Parsed schema:")
parsed_stream.printSchema()

Parsed schema:
root
 |-- key: string (nullable = true)
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: integer (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: integer (nullable = true)
 |-- tip_amount: integer (nullable = true)
 |-- tolls_amount: integer (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: integer (nullable = true)
 |-- congestion_surcharge: integer (nullable = true)
 |-- airport_fee: integer (nullable = true)
 |-- cbd_congestion_fee: integer (nullable = true)
 |-- partition: integer (nu

In [6]:
import time
for q in spark.streams.active:
    if q.name == "raw_events":
        q.stop()

raw_query = (
    parsed_stream.writeStream
    .format("memory")
    .queryName("raw_events")          # table name for spark.sql()
    .outputMode("append")
    .trigger(processingTime="3 seconds")
    .start()
)

time.sleep(8)   # wait for two trigger cycles
print("Messages received so far:")
spark.sql("""
    SELECT key, VendorID, tpep_pickup_datetime, tpep_dropoff_datetime, passenger_count, trip_distance
    FROM raw_events
    ORDER BY partition, offset
""").show(truncate=False)

Messages received so far:
+---+--------+--------------------+---------------------+---------------+-------------+
|key|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|
+---+--------+--------------------+---------------------+---------------+-------------+
|1  |1       |2025-01-01T00:18:38 |2025-01-01T00:26:59  |1.0            |1.6          |
|1  |1       |2025-02-01T00:06:09 |2025-02-01T00:11:51  |0.0            |0.4          |
|1  |1       |2025-01-01T00:32:40 |2025-01-01T00:35:13  |1.0            |0.5          |
|1  |1       |2025-02-01T00:15:13 |2025-02-01T00:20:19  |0.0            |0.7          |
|1  |1       |2025-01-01T00:44:04 |2025-01-01T00:46:01  |1.0            |0.6          |
|1  |1       |2025-02-01T00:15:18 |2025-02-01T00:26:00  |1.0            |2.2          |
|1  |1       |2025-01-01T00:14:47 |2025-01-01T00:16:15  |0.0            |0.4          |
|1  |1       |2025-02-01T00:43:54 |2025-02-01T00:54:17  |1.0            |1.4          |
|1  |1

In [7]:
# incase you already have checkpoints, this will remove them, also wait a moment before excecuting next cell
import shutil
shutil.rmtree("checkpoints/bronze", ignore_errors=True)

In [8]:
bronze_df = raw_stream.selectExpr(
    "CAST(value AS STRING) AS value",
    "timestamp AS kafka_time",
    "topic",
    "partition",
    "offset"
)

bronze_query = (
    bronze_df.writeStream
        .format("iceberg")
        .outputMode("append")
        .option("checkpointLocation", "checkpoints/bronze")
        .toTable("lakehouse.taxi.bronze")
)

In [9]:
import time
time.sleep(5)

spark.sql("""SELECT topic, COUNT(*) FROM lakehouse.taxi.bronze GROUP BY topic""").show()


+-------------------+--------+
|              topic|count(1)|
+-------------------+--------+
| taxi-trips-january|     432|
|taxi-trips-february|     500|
+-------------------+--------+



If the Bronze table shows 0 rows, delete the checkpoint and restart the Bronze stream. Kafka sometimes requires a fresh offset reset when using subscribePattern.

In [10]:
spark.sql("SELECT count(*) FROM lakehouse.taxi.bronze").show()

+--------+
|count(1)|
+--------+
|    1165|
+--------+



In [11]:
spark.sql("SELECT * FROM lakehouse.taxi.bronze LIMIT 3").show()

+--------------------+--------------------+------------------+---------+------+
|               value|          kafka_time|             topic|partition|offset|
+--------------------+--------------------+------------------+---------+------+
|{"VendorID": 1, "...|2026-04-05 21:19:...|taxi-trips-january|        0|     0|
|{"VendorID": 1, "...|2026-04-05 21:19:...|taxi-trips-january|        0|     1|
|{"VendorID": 1, "...|2026-04-05 21:19:...|taxi-trips-january|        0|     2|
+--------------------+--------------------+------------------+---------+------+



# Silver Layer — Cleaning & Enrichment

- Read from the bronze table (or directly from the stream).
- Parse and cast types correctly (timestamps, numeric fields).
- Apply cleaning rules: handle nulls, invalid values, duplicates. Document the rules.
- Enrich with the zone lookup table (pickup and dropoff zone names).
- Write to a silver Iceberg table.

Parsing raw events (memory table)  — 
This temporary query parses the Kafka JSON into columns and writes it to an in‑memory table so we can verify that messages are arriving correctly.

In [12]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json, to_timestamp

silver_schema = StructType([
    StructField("VendorID", IntegerType()),
    StructField("tpep_pickup_datetime", StringType()),
    StructField("tpep_dropoff_datetime", StringType()),
    StructField("passenger_count", IntegerType()),
    StructField("trip_distance", DoubleType()),
    StructField("RatecodeID", IntegerType()),
    StructField("store_and_fwd_flag", StringType()),
    StructField("PULocationID", IntegerType()),
    StructField("DOLocationID", IntegerType()),
    StructField("payment_type", IntegerType()),
    StructField("fare_amount", DoubleType()),
    StructField("extra", DoubleType()),
    StructField("mta_tax", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("tolls_amount", DoubleType()),
    StructField("improvement_surcharge", DoubleType()),
    StructField("total_amount", DoubleType()),
    StructField("congestion_surcharge", DoubleType()),
    StructField("Airport_fee", DoubleType()),
    StructField("cbd_congestion_fee", DoubleType())
])


In [13]:
from pyspark.sql.functions import broadcast

zones_pu = zones.select(
    col("LocationID").alias("PU_id"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
)

zones_do = zones.select(
    col("LocationID").alias("DO_id"),
    col("Zone").alias("dropoff_zone"),
    col("Borough").alias("dropoff_borough")
)


In [14]:
spark.sql("""
CREATE TABLE IF NOT EXISTS lakehouse.taxi.silver (
    VendorID INT,
    passenger_count INT,
    trip_distance DOUBLE,
    RatecodeID INT,
    store_and_fwd_flag STRING,
    PULocationID INT,
    DOLocationID INT,
    payment_type INT,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    congestion_surcharge DOUBLE,
    Airport_fee DOUBLE,
    cbd_congestion_fee DOUBLE,
    pickup_ts TIMESTAMP,
    dropoff_ts TIMESTAMP,
    pickup_zone STRING,
    pickup_borough STRING,
    dropoff_zone STRING,
    dropoff_borough STRING
)
USING iceberg
PARTITIONED BY (days(pickup_ts))
""")


DataFrame[]

Silver cleaning rules  — 
This applies data‑quality filters:
invalid timestamps, zero/negative distances, negative fares, and unrealistic passenger counts are removed.
Duplicates are dropped based on key trip attributes.

In [15]:
from pyspark.sql.functions import col, from_json, to_timestamp, broadcast

bronze_batch = spark.read.format("iceberg").load("lakehouse.taxi.bronze")

parsed_batch = (
    bronze_batch
        .select(from_json(col("value"), silver_schema).alias("d"))
        .select("d.*")
)

typed_batch = (
    parsed_batch
        .withColumn("pickup_ts", to_timestamp("tpep_pickup_datetime"))
        .withColumn("dropoff_ts", to_timestamp("tpep_dropoff_datetime"))
        .drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
)

cleaned_batch = typed_batch.filter(
    "pickup_ts IS NOT NULL AND dropoff_ts IS NOT NULL"
)


Silver enrichment (zone joins) — This joins the cleaned trip data with the zone lookup table to add human‑readable pickup and dropoff zone and borough names.

In [16]:
silver_batch = (
    cleaned_batch
        .join(
            broadcast(zones_pu),
            cleaned_batch.PULocationID == zones_pu.PU_id,
            "left"
        )
        .join(
            broadcast(zones_do),
            cleaned_batch.DOLocationID == zones_do.DO_id,
            "left"
        )
        .drop("PU_id", "DO_id")
)


In [17]:
silver_batch.writeTo("lakehouse.taxi.silver").append()


In [18]:
bronze_stream = (
    spark.readStream
        .format("iceberg")
        .load("lakehouse.taxi.bronze")
)

In [19]:
parsed_stream = (
    bronze_stream
        .select(from_json(col("value"), silver_schema).alias("d"))
        .select("d.*")
        .withColumn("pickup_ts", to_timestamp("tpep_pickup_datetime"))
        .withColumn("dropoff_ts", to_timestamp("tpep_dropoff_datetime"))
        .filter("pickup_ts IS NOT NULL AND dropoff_ts IS NOT NULL")
        .dropDuplicates([
        "VendorID",
        "pickup_ts",
        "dropoff_ts",
        "PULocationID",
        "DOLocationID",
        "total_amount"
    ])
)

In [20]:
silver_stream = (
    parsed_stream
        .join(broadcast(zones_pu), col("PULocationID") == col("PU_id"), "left")
        .join(broadcast(zones_do), col("DOLocationID") == col("DO_id"), "left")
        .drop("PU_id", "DO_id")
)

Write Silver table — This writes the cleaned and enriched stream into the Silver Iceberg table with checkpointing.

In [21]:
silver_query = (
    silver_stream.writeStream
        .format("iceberg")
        .outputMode("append")
        .option("checkpointLocation", "checkpoints/silver")
        .toTable("lakehouse.taxi.silver")
)

In [22]:
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.silver").show()


+--------+
|count(1)|
+--------+
|    3262|
+--------+



In [23]:
spark.sql("SELECT * FROM lakehouse.taxi.silver LIMIT 10").show(truncate=False)

+--------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+-------------------+-------------------+-----------------------------+--------------+-------------------+---------------+
|VendorID|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|pickup_ts          |dropoff_ts         |pickup_zone                  |pickup_borough|dropoff_zone       |dropoff_borough|
+--------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+------------------

In [ ]:
spark.sql("SELECT COUNT(*) FROM lakehouse.taxi.silver").show()
spark.sql("""
    SELECT 
    PULocationID,
    pickup_zone,
    pickup_borough,
    DOLocationID,
    dropoff_zone,
    dropoff_borough
FROM lakehouse.taxi.silver
LIMIT 10;
""").show(truncate=False)

+--------+
|count(1)|
+--------+
|    3262|
+--------+



In [ ]:
spark.sql("SELECT * FROM lakehouse.taxi.silver LIMIT 10;").show()

# Gold Layer — Aggregation

- Produce at least one meaningful aggregation from the silver data.
- Example: hourly trip counts and average fare by pickup zone.
- Write to a gold Iceberg table with a justified partitioning strategy.

Gold aggregation — This block computes hourly trip counts and average fares per pickup zone using a watermark to handle late data.

Create Gold table — This creates the Gold Iceberg table, partitioned by day for efficient time‑based queries.

Write Gold stream — This writes the aggregated results into the Gold table as a continuous streaming query.

In [ ]:
from pyspark.sql.functions import window, count, avg, col, to_timestamp

silver_df = spark.read.format("iceberg").load("lakehouse.taxi.silver")

agg_batch = (
    silver_df
        .withColumn("pickup_ts", to_timestamp("pickup_ts"))
        .groupBy(
            window(col("pickup_ts"), "1 hour").alias("w"),
            col("pickup_zone")
        )
        .agg(
            count("*").alias("trip_count"),
            avg("fare_amount").alias("avg_fare")
        )
        .select(
            col("pickup_zone"),
            col("w.start").alias("pickup_hour"),
            "trip_count",
            "avg_fare"
        )
)

agg_batch.write.format("iceberg").mode("overwrite").saveAsTable("lakehouse.taxi.gold")

In [ ]:
spark.streams.active


In [ ]:
spark.sql("""
SELECT pickup_hour, pickup_zone, trip_count, avg_fare
FROM lakehouse.taxi.gold
ORDER BY pickup_hour, pickup_zone
LIMIT 20
""").show(truncate=False)


In [ ]:
spark.sql("SELECT * FROM lakehouse.taxi.gold LIMIT 20").show()


In [ ]:
spark.sql("""SELECT *
FROM lakehouse.taxi.gold
WHERE pickup_hour >= '2025-01-01'
  AND pickup_hour < '2025-01-03'"""
  ).show()

In [ ]:
spark.sql("""SELECT topic, COUNT(*) AS cnt
FROM lakehouse.taxi.bronze
GROUP BY topic;"""
  ).show()

In [ ]:
#Restart check
spark.sql("""
    SELECT COUNT(*) FROM lakehouse.taxi.bronze;"""
      ).show()
spark.sql("""        
    SELECT COUNT(*) FROM lakehouse.taxi.silver;"""
      ).show()
spark.sql("""
    SELECT COUNT(*) FROM lakehouse.taxi.gold;"""
     ).show()
